# Inference

The full pipeline on the test stacks in `input/test/`, with the models trained
on this machine: two detectors, their merged node set, NETr embeddings, the edge
transformer, post-processing, the division classifier, and the submission CSV.

Run it with the repository's environment (`uv sync`, then the `.venv` kernel).
Every step writes to `output/inference/`.

## Setup

In [ ]:
import os
import sys
import json
from pathlib import Path

import pandas as pd
import torch

# Every entry point runs from src/, and the paths below are relative to it.
REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
os.chdir(REPO / "src")
sys.path.insert(0, ".")

PY = sys.executable
N_GPUS = max(torch.cuda.device_count(), 1)
# --local-addr: this host's FQDN does not resolve, so torchrun's default lookup
# hangs when it hands the address to the workers.
TORCHRUN = f"{PY} -m torch.distributed.run --standalone --local-addr=127.0.0.1 --nproc_per_node={N_GPUS}"

DATA_DIR = "../input/test/"
OUT = "../output/inference/"
os.makedirs(OUT, exist_ok=True)

## Configuration

In [ ]:
# The shipped models, trained on this machine (the same files as the Kaggle
# datasets theoviel/biohub-weights-2 and biohub-weights-3).
TRACK_W_BASE  = "../logs/2026-09-13/1/"
TRACK_W_ADD   = "../logs/2026-09-24/5/"
CONTRASTIVE_W = "../logs/contrastive/2026-09-09/1/"
EDGE_W        = "../logs/edge_run/2026-09-24/4/"
DIVISION_W    = "../logs/division/2026-09-16/0/"

# Or the models of the latest ./train.sh run, from logs/pipeline_runs.env.
USE_RETRAINED = False
if USE_RETRAINED:
    runs = dict(l.strip().split("=", 1) for l in open("../logs/pipeline_runs.env") if "=" in l)
    TRACK_W_BASE, TRACK_W_ADD, CONTRASTIVE_W, EDGE_W, DIVISION_W = (
        runs[k] + "/" for k in ("DET", "DET_ADD", "NETR", "EDGE", "DIV"))

USE_DIVISION_CLS = True
DIVISION_TAU = 0.5

In [ ]:
from inference_contrastive import embeddings_key

# The edge model reads the embeddings under the key its NETr run wrote them, and
# records the node folder it was fitted on: check both match the models above.
edge_args = json.load(open(EDGE_W + "args.json"))
EMB_KEY = edge_args.get("embeddings_key") or "embeddings"
assert EMB_KEY == embeddings_key(CONTRASTIVE_W), (
    f'EDGE_W reads "{EMB_KEY}", but CONTRASTIVE_W writes "{embeddings_key(CONTRASTIVE_W)}"')
assert "v4_" in (edge_args.get("nodes_folder") or ""), (
    f"EDGE_W was trained on {edge_args.get('nodes_folder')!r}, not a v4 node set; "
    "the merge parameters below will not match its node set")

# fold_fullfit, not the fold models: a test stack is out of fold for every fold
# model, and the fullfit is trained on both embryos under the same recipe.
for _w in (TRACK_W_BASE, TRACK_W_ADD, CONTRASTIVE_W, EDGE_W) + ((DIVISION_W,) if USE_DIVISION_CLS else ()):
    assert os.path.exists(_w + "fold_fullfit/ckpt_final.pt"), f"no fold_fullfit/ckpt_final.pt under {_w}"

# Detection params, per detector.  NMS radius is shared: the merge distance is
# only meaningful against a node set already suppressed at it.
NMS_RADIUS_UM = 6.0
DET_THRESHOLD_BASE = 0.45
DET_THRESHOLD_ADD  = 0.40

# Merge.  An ADD detection is kept only if no BASE detection lies within
# D_ADD_UM of it, in the same frame.  D_ADD_UM must clear NMS_RADIUS_UM:
# appending inside the suppression radius recreates exactly the pair the base
# model's own NMS removed.  Measured flat from 7 to 12 um, so 7 is the floor
# rather than a tuned value.
D_ADD_UM = 7.0

EDGE_THRESHOLD = 0.45
MAX_NODES = 1500

## 1. Detection — both detectors

In [ ]:
# Two independent detection passes. Each detector runs at its own threshold and
# the same NMS radius.
for tag, weights, thr in (
    ("base", TRACK_W_BASE, DET_THRESHOLD_BASE),
    ("add",  TRACK_W_ADD,  DET_THRESHOLD_ADD),
):
    cmd = (
        f"{TORCHRUN} infer_detector.py"
        f" --data_dir {DATA_DIR}"
        f" --log_folder {weights}"
        f" --det_threshold {thr}"
        f" --nms_radius_um {NMS_RADIUS_UM}"
        f" --out_dir {OUT}preds_{tag}/"
    )
    print(f"\n=== {tag}: {weights}\n{cmd}\n")
    !{cmd}

## 2. Merge the two node sets  *(the ensemble)*

`N = BASE(th_base) ∪ { a ∈ ADD(th_add) : dist(a, BASE) > d_add }`, per frame.

ADD only fills gaps BASE missed; it never overrides or displaces a BASE
detection, so every surviving node is a real detector peak rather than a blend
of two. A consensus/centroid-averaging variant was tried and scored *worse*
end to end at every threshold, despite scoring better on an offline proxy.

`scripts/merge_detectors.py` is the merge that built the edge model's training
node set.

In [ ]:
cmd = (
    f"{PY} ../scripts/merge_detectors.py"
    f" --base {OUT}preds_base --add {OUT}preds_add"
    f" --cfg {DET_THRESHOLD_BASE} {DET_THRESHOLD_ADD} {D_ADD_UM}"
    f" --out {OUT}preds/"
)
print(cmd, "\n")
!{cmd}

## 3. Contrastive embeddings (NETr)

One appearance vector per detection, written **back into the detection npz**
rather than into a folder of its own, so the coordinates and their embeddings
cannot drift apart. The edge model reads them as node features; without them it
falls back to geometry alone and scores materially worse.

In [ ]:
# --fold fullfit runs the one model over every stack: the out-of-fold default
# has no meaning on a test split whose embryos are not the training ones.
cmd = (
    f"{TORCHRUN} inference_contrastive.py"
    f" --exp_folder {CONTRASTIVE_W}"
    f" --coords_log_folder {OUT}preds/"
    f" --data_dir {DATA_DIR}"
    " --fold fullfit"
    f" --key {EMB_KEY}"
)
print(cmd, "\n")
!{cmd}

## 4. Edge transformer

Scores every candidate link, and saves **all** of them alongside the committed
ones. That is what makes the threshold and the linking rule a post-processing
decision instead of a GPU one — the next cell re-solves the assignment from this
array, so the threshold here does not reach the final graph.

In [ ]:
cmd = (
    f"{TORCHRUN} infer_edge.py"
    f" --log_folder {EDGE_W}"
    f" --nodes_folder {OUT}preds/"
    f" --data_dir {DATA_DIR}"
    f" --out_dir {OUT}preds_edge/"
    f" --threshold {EDGE_THRESHOLD}"
    f" --embeddings_key {EMB_KEY}"
    f" --max_nodes {MAX_NODES}"
)
print(cmd, "\n")
!{cmd}

## 5. Edge post-processing

`main_edge_postprocess.py`: ILP node filter, assignment, gap recovery,
divisions and short-track pruning, with every knob passed explicitly rather
than left to its `DEFAULTS`. `--no_score` because the test split has no ground
truth.

In [ ]:
# Post-processing. Two chains, because the division gate needs a different
# proposal pool than the ungated submission does -- and they are mutually
# exclusive, so the flag picks one.
#
# USE_DIVISION_CLS = False -- the `_pp_fall_tuned_ilp` knobs swept on all folds
#   of logs/edge_run/2026-09-24/4. Worth +0.0031 over the script defaults:
#
#     default knobs   score 0.8760   adj 0.8631   division 0.1283
#     these knobs     score 0.8791   adj 0.8643   division 0.1475
#
# USE_DIVISION_CLS = True -- harvest the ILP's *own* division forks instead,
#   with both caps opened and the divergence gate off, and let the classifier
#   in the next cell do the rejecting. The tuned knobs above tune the geometric
#   proposer, which this replaces, so they do not apply here.
#
#     harvest + gate  score 0.8860   adj 0.8620   division 0.2398
#
#   Measured out of fold on the same 199 stacks, each embryo scored by the
#   model that held it out. Nearly all of the +0.0069 over the tuned branch is
#   the division term (0.1475 -> 0.2398); adj gives back 0.0023.
#
# The harvest is only ever correct followed by the gate. On its own it scores
# 0.8531 -- *below* running with no divisions at all (0.8638) -- because it
# finds 81 of 151 divisions and brings 2362 false ones with them.
if USE_DIVISION_CLS:
    # `recover_max_gap 2` is the one --fold all knob that carries over: it is
    # gap recovery, which the harvest does not touch. The other two that sweep
    # moved (div_min_prob 0.325, div_symmetry_tau 0.525) are division gates,
    # and applying them here prunes the ILP's forks *before* the classifier
    # sees them -- measured, that costs 0.0018 and takes 6bba from 45 true
    # divisions to 27:
    #
    #   harvest, no tuned knobs        0.8855   adj 0.8616   div 0.2396
    #   harvest + recover_max_gap 2    0.8860   adj 0.8620   div 0.2398
    #   harvest + all three            0.8842   adj 0.8633   div 0.2089
    PP_ARGS = (
        " --recover_max_gap 2"
        " --recover_max_residual 5.0"
        " --ilp_div_forks"
        " --div_no_divergence"
        " --div_frame_frac_cap 1.0"
        " --div_global_frac_cap 1.0"
    )
else:
    PP_ARGS = (
        " --recover_max_gap 2"
        " --recover_max_residual 5.0"
        " --recover_min_dup 3.0"
        " --linefit 0.6"
        " --linefit_window 3"
        " --vel_weight 0.02"
        " --vel_momentum 0.7"
        " --div_min_prob 0.325"
        " --div_max_parent_um 11.0"
        " --div_max_existing_child_um 10.0"
        " --div_max_sister_um 14.0"
        " --div_symmetry_tau 0.525"
        " --div_diverge_um 2.5"
        " --div_frame_frac_cap 0.0076"
        " --div_global_frac_cap 0.00375"
    )

cmd = (
    f"{PY} main_edge_postprocess.py"
    f" --preds_dir {OUT}preds_edge/"
    f" --out_dir {OUT}preds_pp/"
    f" --geff_dir {OUT}preds_pp_geff/"
    " --threshold 0.40"
    " --ilp"
    " --ilp_threshold 0.40"
    " --ilp_disappearance 0.75"
    " --ilp_division 1.0"
    " --min_track 5"
    + PP_ARGS +
    " --no_score"
    " --jobs 8"
)
print(cmd, "\n")
!{cmd}

## 6. Division classifier gate  *(USE_DIVISION_CLS)*

The only stage besides the detectors that reads pixels. It links nothing: it
scores each fork the chain above proposed — a 32×32 crop over 3 frames centred
on the parent — and deletes the second-child edge of the ones it disbelieves.
Deleting a second child cannot break a track, so the edge metric has almost
nothing to lose.

When the flag is off this cell is a passthrough and `GEFF_DIR` stays on the
post-processing output.

In [ ]:
# Everything downstream reads GEFF_DIR, so the flag changes one variable rather
# than the submission cell.
GEFF_DIR = f"{OUT}preds_pp_geff/"

if USE_DIVISION_CLS:
    cmd = (
        f"{PY} division_cls/apply_fork_gate.py"
        f" --preds_dir {OUT}preds_pp/"
        f" --data_dir {DATA_DIR}"
        f" --run_dir {DIVISION_W}"
        f" --out_dir {OUT}preds_pp_gated/"
        f" --geff_dir {OUT}preds_pp_gated_geff/"
        f" --tau {DIVISION_TAU}"
        " --fold fullfit"
        # Not a boolean -- this is the min_track to re-prune at, and it
        # must match the chain above. prune_short ran while every
        # harvested fork was still in the graph, so a fragment held up by
        # a fork the classifier then deletes is left stranded with its own
        # edges, which score as false positives.
        " --reprune 5"
    )
    print(cmd, "\n")
    !{cmd}
    GEFF_DIR = f"{OUT}preds_pp_gated_geff/"

print("-> submission reads:", GEFF_DIR)

## Submission

In [ ]:
!{PY} submission.py {GEFF_DIR} -o {OUT}submission.csv

In [ ]:
sub = pd.read_csv(f"{OUT}submission.csv")
sub.sample(5)